# Introduction
In the world of hospitality, customer reviews are a goldmine of insights — but with thousands of reviews scattered across text, voice, video, and images, it's hard for businesses to make sense of them all.

An intelligent pipeline that takes in text/audio/video reviews and outputs a structured, grounded summary — powered by RAG, embeddings, few-shot prompting, image/video/audio understanding.

This project builds a Smart Review Summarizer that:

# Phase 1 (By April 20th ) - Gen AI capabilities incorporated
1. Structured output/JSON model: Current program is storing the JSON structure in a output file.
2. Document understanding : The program currently reads the reviews from the sample data created in the the excel file.
3. Embeddings : Sentence transformers are used in this project - all-miniLM-L6-v2
4. Retrieval augmented genration (RAG) : Using FLAN-T5 model for genrating relevant summaries by retriving passges from the vector store based on
   the query
5. Vector store : FAISS index
6. Few shot prompt design
7. Classification of reviews

# Phase 2 (After Apr 20th)
1. Using agents to interact with live reviews using Google API
2. Using Context Cache
3. MLOps : Making the project production mode


# GitHub

https://github.com/prakashpillai/LearningGenAI.git




# Use Case
Imagine a hotel manager trying to understand guest feedback from Booking.com, YouTube video reviews, and voice messages. Instead of reading hundreds of reviews manually, our system:

Summarizes guest experiences (cleanliness, location, service, value, etc.)
Categorizes pros/cons from real reviews
Grounds summaries in similar past reviews using RAG




# Step 1# Creating a sample Dataset
Creating a simple dataset of reviews using Excel. Steps followed for the same

Created a simple excel file
uploaded the file on right side by clicking upload in the dataset.reading the excel using Pandas.

In [1]:
import pandas as pd

#loading sample data from file in the pandas dataframe.
df = pd.read_excel('/kaggle/input/test-data/Dataset_Kaggle_project.xlsx', engine='openpyxl')

df.head()


,Review,Scoring
0,Absolutely fantastic,5
1,"Absolutely magical, our grandchildren were tot...",5
2,After spending hours standing in the long over...,1
3,Best place on earth,5
4,Cool,5


# Step 2# Preprocess & Embed reviews

In [2]:
!pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 58.8 MB/s eta 0:00:00:00:0100:01


Generate text embeddings for each review using pre-trained model and store them in a FAISS vector database.

In [3]:
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np
import json

In [4]:
#using the light weight yet powerful sentence embedding model 'all-miniLM-L6-V2' to embed the reviews
embed_model = SentenceTransformer('all-miniLM-L6-v2')
#converting Pandas dataframe to list.
texts = df['Review'].tolist()
#generate embeddings
embeddings = embed_model.encode(texts,show_progress_bar=True)
# The review text are stored as vector in 384 dimensions.This embedings will passed into FAISS index.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

# Step 3# Create Vector Store (FAISS index)

In [5]:
#Initializing and create the FAISS index

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
# This adds the review embeddings to FAISS index
index.add(embeddings)

# Step 4# RAG setup (Retrieval Augmented generation) 

RAG Combines 
1. Retrieval (Search from vector store) 
2. Generation(use the language model to answer based on retrieved data)

In [6]:
#previously we used the embed model to convert the reviews into numeric form
#then convert the query into vector form
#find the closest match using FAISS . Now going to use the Generative model(FLAN-T5) 
#which will assist to perform the text summarization
#based on the simple query

#Import FLAN-T5 model from Hugging Face.
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "google/flan-t5-small"
#converts the text into tokens that model understands.
tokenizer = T5Tokenizer.from_pretrained(model_name)
#T%ConditionalGenration is the actual FLAN-T5 model, used for tasks like summarization or Q & A.
model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

# Step 5# Few shot prompt Design

In [ ]:
#just putting some string.
query ='Disneyland Parks'
# When you generate embbedings using "SentenceTransformer" they return as NumpY array by default. 
# Converting our string to a query vector. embed_model has all the reviews which we shared through dataset.
query_vector = embed_model.encode([query], convert_to_numpy=True).reshape(1, -1)

#previously we used the embed model to convert the review into numeric form
#convert the query into vector form
#find the closest match using FAISS . Now going to use the Generative model(FLAN-T5) which will assist to give answers
#based on the simple query

#Search for top 3 similar reviews
k = 3
distances , indices = index.search(query_vector,k)
# The row numbers of the most similar reviews
print(indices)
#how close those reviews are to the query
print(distances)
# This will take values of 0,2,3 indices from DF and store in top_k_reviews as a list.
top_k_reviews =  [df.iloc[idx]['Review'] for idx in indices[0]]
print(top_k_reviews)

# === Define full RAG to JSON function ===

def rag_to_json(query, k):
    # Retrieve top documents
    query_vector = embed_model.encode([query], convert_to_numpy=True).reshape(1, -1)
    distances, indices = index.search(query_vector, k)
    top_k_reviews = [df.iloc[idx]['Review'] for idx in indices[0]] 

# Prepare context from reviews
    context = " ".join([review[:300] for review in top_k_reviews])
    
# Generate answer using FLAN-T5
    prompt = f"Summarize the following  hotel reviews : {context}"
    inputs = tokenizer(prompt,return_tensors='pt',truncation=True, max_length=512)
    outputs = model.generate(**inputs,max_new_tokens=100)

    summary = tokenizer.decode(outputs[0],skip_special_tokens=True)
    
    print(f"Answer :{summary}")
      # Format as JSON
    result = {
        "query": query,
        "summary": summary,
        "top_k_reviews": top_k_reviews
        }
    #print(json.dumps(result, indent=2, ensure_ascii=False))
    
    return result
    
output=rag_to_json("Disneyland Parks",3)

# Step 6# JSON Summary Generation

Exporting the JSON to a external file name ''

In [ ]:
# Save the output from the prev step and save as a JSON file
with open("summary_output.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("### Summary exported to 'summary_output.json' successfully ####")

# Phase 2 - Future product enchancments
1. To review the images.
2. Use Agents to interact with live reviews using API
3. Adding the MLOps to make the product